# Support Ticket Classification: Feature Engineering + MLP, BERT and GPT

Complete these activities in order:

- Inspect the shared ticket data and train/validation/test splits.
- Train a word-count MLP, fine-tune DistilBERT and fine-tune GPT-2.
- Compare the three methods, inspect errors and save models for new tickets.

[Open in Colab](https://colab.research.google.com/github/wlos/AI_SS/blob/main/practice/session14-15/Session14-15%20Support%20Ticket%20Classification%20Practice.ipynb)

Select a GPU runtime for Transformer fine-tuning, then run the notebook from top to bottom. The first code cell installs the required packages. The data cell downloads the [ticket CSV](https://raw.githubusercontent.com/wlos/AI_SS/main/sample_data/support_tickets.csv) into `sample_data/` if it is missing. Model files are written to `transformer/` in the current working directory.

In Colab, download any saved models you want to keep before the runtime is deleted.


In [ ]:
import subprocess
import sys

# Install the libraries before importing them in later cells.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "torch>=2.6,<3", "transformers==4.57.1", "datasets>=3,<5", "accelerate>=1,<2",
    "pandas", "scikit-learn", "matplotlib", "joblib",
])


## 1. Data: Support Tickets and Shared Splits

The supplied CSV contains 80 fictional teaching tickets, with 20 in each class and explicit train/validation/test splits. It is deliberately small and balanced. It cannot estimate real service-desk performance, and short synthetic wording can make classification easier than realistic tickets.

| Category | Routing rule | Example |
|---|---|---|
| Account access | Credentials, locked accounts and authentication | My account is locked. |
| Hardware | Physical device faults | The laptop screen is cracked. |
| Software | Application installation, crashes and behaviour | The spreadsheet application crashes. |
| Network | Connectivity and network access | The campus Wi-Fi disconnects. |

A real ticket can involve several issues. This exercise assigns one primary category; ambiguous cases should be reviewed rather than forcing an unjustified label.

[Download the supplied ticket CSV](https://raw.githubusercontent.com/wlos/AI_SS/main/sample_data/support_tickets.csv).


In [ ]:
import os
from urllib.request import urlretrieve
import pandas as pd

TICKET_CSV = "sample_data/support_tickets.csv"
TICKET_CSV_URL = "https://raw.githubusercontent.com/wlos/AI_SS/main/sample_data/support_tickets.csv"
os.makedirs("sample_data", exist_ok=True)
if not os.path.isfile(TICKET_CSV):
    urlretrieve(TICKET_CSV_URL, TICKET_CSV)  # Download the shared data once per runtime.
tickets = pd.read_csv(TICKET_CSV)  # Open the supplied labelled data when first used.
print(tickets.groupby(["split", "category"]).size().unstack(fill_value=0))
print(tickets.head(8).to_string(index=False))
assert not tickets["text"].str.lower().duplicated().any(), "Check duplicate ticket text before training."

class_names = ["Account access", "Hardware", "Software", "Network"]
label_to_id = {name: number for number, name in enumerate(class_names)}
id_to_label = {number: name for name, number in label_to_id.items()}
tickets["label"] = tickets["category"].map(label_to_id)
train_tickets = tickets.loc[tickets["split"] == "train"].copy()
validation_tickets = tickets.loc[tickets["split"] == "validation"].copy()
test_tickets = tickets.loc[tickets["split"] == "test"].copy()


**Choosing the methods**

Implement three approaches on the same labelled tickets:

| Method | Input representation | Classifier |
|---|---|---|
| Feature Engineering + MLP | Fixed word-count features | A scikit-learn MLP trained from scratch |
| BERT | Learned bidirectional contextual features | Fine-tuned DistilBERT with a classification head |
| GPT | Learned causal contextual features | Fine-tuned GPT-2 with a classification head |

Run the sections in order: all methods share the data splits and label mapping defined above. Use validation data to select weights and reserve test data for the final comparison.

This extends the count-feature idea in [Sessions 5–6, Section 5](https://wlos.github.io/AI_SS/Session5-6%20AI%20Opportunities.html#5-ai-for-multi-label-text-classification). Here a small fully connected neural network replaces logistic regression. Each ticket has one category, so this is single-label multiclass classification.


## 2. Feature Engineering + MLP

**Ticket text → word-count features → MLP → category.** The feature engineering step defines a bag-of-words representation; the MLP learns how those features relate to the labels.

`CountVectorizer` builds a vocabulary from **training text only**. Each ticket becomes a sparse vector of word counts. `MLPClassifier` provides a fully connected hidden layer with ReLU activation and an output layer for the four categories. Its `fit()` method handles optimisation internally, so we do not write a PyTorch training loop.

Try hidden layers with 16 and 32 units, train each on the training split, and select the model with the higher validation macro F1. `random_state` makes each initialisation reproducible. The `lbfgs` solver suits this small dataset. Keep the count matrices sparse; counts still ignore word order and context.

See the [scikit-learn MLPClassifier documentation](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html).

The package installation cell at the start of this notebook includes the libraries for this method.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

SEED = 42
count_vectorizer = CountVectorizer(lowercase=True, ngram_range=(1, 1))
# Fit the vocabulary on training tickets only; reuse it for validation.
train_counts = count_vectorizer.fit_transform(train_tickets["text"])
validation_counts = count_vectorizer.transform(validation_tickets["text"])
y_count_train = train_tickets["label"].to_numpy()
y_count_validation = validation_tickets["label"].to_numpy()

print("Vocabulary size:", len(count_vectorizer.vocabulary_))
print("Training input shape:", train_counts.shape)
print("First ticket:", train_tickets.iloc[0]["text"])
first_counts = train_counts[0].toarray()[0]  # Densify one row only for inspection.
print([(word, int(first_counts[index])) for index, word in enumerate(count_vectorizer.get_feature_names_out()) if first_counts[index] > 0])


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

COUNT_HIDDEN_UNITS = (16, 32)
count_model = None
best_count_macro_f1 = -1.0
count_validation_rows = []

for hidden_units in COUNT_HIDDEN_UNITS:
    candidate = MLPClassifier(
        hidden_layer_sizes=(hidden_units,), activation="relu", solver="lbfgs",
        alpha=0.01, max_iter=1000, random_state=SEED,
    )
    candidate.fit(train_counts, y_count_train)  # Learn from training tickets only.
    validation_predictions = candidate.predict(validation_counts)
    validation_macro_f1 = f1_score(
        y_count_validation, validation_predictions,
        labels=list(id_to_label), average="macro", zero_division=0,
    )
    count_validation_rows.append({
        "hidden_units": hidden_units,
        "accuracy": accuracy_score(y_count_validation, validation_predictions),
        "macro_f1": validation_macro_f1,
    })
    if validation_macro_f1 > best_count_macro_f1:
        best_count_macro_f1 = validation_macro_f1
        count_model = candidate  # Retain the fitted model selected on validation data.

print(pd.DataFrame(count_validation_rows).to_string(index=False))
print("Selected hidden layer:", count_model.hidden_layer_sizes)


## 3. BERT: Fine-Tuning DistilBERT

**Ticket text → token IDs → bidirectional encoder → classification head → category.** DistilBERT is the smaller BERT-family model used for this implementation. Unlike word counts, its learned features depend on token order and context.

**Prepare inputs and the task head**

Each ticket label is one integer identifying its category. DistilBERT supplies pretrained language features, and a newly initialised classification head supplies four task scores. The initialisation warning for the new head is expected: its categories have not been learned yet.

Padding makes batch lengths compatible; its attention mask marks real tokens. Truncation limits input length and can remove useful details from long tickets.


The package installation cell at the start of the notebook includes Transformers 4.57.1, Datasets and Accelerate for both Transformer methods. GPT-2 requires more memory than DistilBERT.

`Trainer` automatically uses CUDA or Apple MPS when available, and CPU otherwise. Each training cell prints its selected device. The scikit-learn MLP uses CPU.

Model caches, training checkpoints and saved classifiers are stored under `transformer/` in the current working directory. The code creates each folder when first needed.


In [ ]:
import os
from datasets import Dataset, DatasetDict

ticket_splits = DatasetDict({
    name: Dataset.from_pandas(frame[["text", "label"]], preserve_index=False)
    for name, frame in [("train", train_tickets), ("validation", validation_tickets), ("test", test_tickets)]
})

from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, set_seed

BERT_CHECKPOINT = "distilbert/distilbert-base-uncased"
TICKET_TOKENIZER_CACHE_DIR = "transformer/ticket-tokenizer-cache"
os.makedirs(TICKET_TOKENIZER_CACHE_DIR, exist_ok=True)  # Create the local tokenizer cache folder if needed.
# Use the tokenizer paired with the pretrained encoder.
ticket_tokenizer = AutoTokenizer.from_pretrained(BERT_CHECKPOINT, cache_dir=TICKET_TOKENIZER_CACHE_DIR)
SEED = 42

MAX_TICKET_TOKENS = 96

def prepare_tickets(batch):
    return ticket_tokenizer(batch["text"], truncation=True, max_length=MAX_TICKET_TOKENS)

encoded_tickets = ticket_splits.map(prepare_tickets, batched=True, remove_columns=["text"])
set_seed(SEED)  # Initialise the new task head reproducibly.
TICKET_MODEL_CACHE_DIR = "transformer/ticket-model-cache"
os.makedirs(TICKET_MODEL_CACHE_DIR, exist_ok=True)
ticket_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_CHECKPOINT, num_labels=len(class_names), cache_dir=TICKET_MODEL_CACHE_DIR,
    id2label=id_to_label, label2id=label_to_id,
)
ticket_collator = DataCollatorWithPadding(tokenizer=ticket_tokenizer)


**Train and validate BERT**

Accuracy counts all correct predictions. Macro F1 gives each category equal weight; a confusion matrix shows where errors occur. Both the encoder and classification head update here. Freezing the encoder would instead train a task head on fixed language features.

Compare against a majority-class baseline. The newly initialised head must learn the ticket labels; pretrained language features alone do not define these categories.


In [ ]:
import os
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from transformers import Trainer, TrainingArguments

def ticket_metrics(prediction):
    logits, labels = prediction
    predicted = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, predicted),
        "macro_f1": f1_score(labels, predicted, labels=list(id_to_label), average="macro", zero_division=0),
    }

TICKET_CHECKPOINT_DIR = "transformer/ticket-checkpoints"
os.makedirs(TICKET_CHECKPOINT_DIR, exist_ok=True)
ticket_args = TrainingArguments(
    output_dir=TICKET_CHECKPOINT_DIR,
    learning_rate=3e-5, per_device_train_batch_size=8, per_device_eval_batch_size=8,
    num_train_epochs=3, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
    logging_steps=5, report_to="none", seed=SEED, dataloader_num_workers=0,
)
print("BERT training device:", ticket_args.device)  # Show the device selected by Trainer.
ticket_trainer = Trainer(
    model=ticket_model, args=ticket_args,
    train_dataset=encoded_tickets["train"], eval_dataset=encoded_tickets["validation"],
    processing_class=ticket_tokenizer, data_collator=ticket_collator, compute_metrics=ticket_metrics,
)
ticket_trainer.train()
print("Validation:", ticket_trainer.evaluate())


## 4. GPT: Fine-Tuning GPT-2

**Ticket text → token IDs → causal decoder → last-token feature → classification head → category.**

This implementation uses the downloadable [GPT-2 checkpoint](https://huggingface.co/openai-community/gpt2). It applies supervised fine-tuning to a decoder-only model, using the same ticket splits and labels as the other methods. A newly initialised classification head produces four category scores; both this head and the pretrained decoder are updated.

GPT-2 was pretrained to predict the next token. Here we use a **sequence classification head**, rather than asking it to generate a label as text. The last non-padding token can attend to all earlier ticket tokens, so its contextual feature can represent the available ticket. There is no separate encoder or cross-attention. This example runs locally and requires no API key.

**Prepare inputs and the task head**

GPT-2 has no default padding token. Reuse its end-of-text token for padding, set the model’s `pad_token_id` to match, and keep the attention mask. Right padding leaves the real ticket tokens at their original positions. See the [GPT-2 sequence classification documentation](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/gpt2#transformers.GPT2ForSequenceClassification).


In [ ]:
import os
GPT_CHECKPOINT = "openai-community/gpt2"
GPT_CACHE_DIR = "transformer/gpt-cache"
os.makedirs(GPT_CACHE_DIR, exist_ok=True)
gpt_tokenizer = AutoTokenizer.from_pretrained(GPT_CHECKPOINT, cache_dir=GPT_CACHE_DIR)
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token  # Reuse an existing token without resizing embeddings.
gpt_tokenizer.padding_side = "right"

def prepare_gpt_tickets(batch):
    return gpt_tokenizer(batch["text"], truncation=True, max_length=MAX_TICKET_TOKENS)

gpt_encoded_tickets = ticket_splits.map(prepare_gpt_tickets, batched=True, remove_columns=["text"])
set_seed(SEED)  # Make the new classification head reproducible.
gpt_model = AutoModelForSequenceClassification.from_pretrained(
    GPT_CHECKPOINT, num_labels=len(class_names), cache_dir=GPT_CACHE_DIR,
    id2label=id_to_label, label2id=label_to_id,
    pad_token_id=gpt_tokenizer.pad_token_id,
)
gpt_model.config.use_cache = False  # Classification does not need a generation cache.
gpt_collator = DataCollatorWithPadding(tokenizer=gpt_tokenizer)


**Train and validate GPT**

Reuse the accuracy and macro F1 function from the BERT method. Use the same token limit, validation split and three training epochs, with a smaller batch size for GPT-2. Select the best checkpoint by validation macro F1. The models differ in size and pretraining, so this is a practical comparison rather than a controlled test of architecture alone.


In [ ]:
import os
GPT_CHECKPOINT_DIR = "transformer/gpt-checkpoints"
os.makedirs(GPT_CHECKPOINT_DIR, exist_ok=True)
gpt_args = TrainingArguments(
    output_dir=GPT_CHECKPOINT_DIR,
    learning_rate=3e-5, per_device_train_batch_size=4, per_device_eval_batch_size=4,
    num_train_epochs=3, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
    logging_steps=5, report_to="none", seed=SEED, dataloader_num_workers=0,
)
print("GPT-2 training device:", gpt_args.device)  # Show the device selected by Trainer.
gpt_trainer = Trainer(
    model=gpt_model, args=gpt_args,
    train_dataset=gpt_encoded_tickets["train"], eval_dataset=gpt_encoded_tickets["validation"],
    processing_class=gpt_tokenizer, data_collator=gpt_collator, compute_metrics=ticket_metrics,
)
gpt_trainer.train()  # Update the decoder and task head using labelled training tickets.
print("GPT-2 validation:", gpt_trainer.evaluate())


## 5. Compare the Three Methods

Use identical test tickets and labels for all three methods. Accuracy and macro F1 measure their predictions; confusion matrices and examples reveal different errors. The majority-class result gives a simple baseline.

The test set contains only 16 fictional tickets. A pretrained Transformer is not guaranteed to beat the smaller NN, and a few changed predictions can substantially affect these scores.


In [ ]:
import matplotlib.pyplot as plt

after_ticket = ticket_trainer.predict(encoded_tickets["test"])
true_labels = after_ticket.label_ids
gpt_test_output = gpt_trainer.predict(gpt_encoded_tickets["test"])
gpt_predictions = gpt_test_output.predictions.argmax(axis=-1)
assert (gpt_test_output.label_ids == true_labels).all(), "Keep test rows aligned across models."
predicted_labels = after_ticket.predictions.argmax(axis=-1)
test_counts = count_vectorizer.transform(test_tickets["text"])
count_predictions = count_model.predict(test_counts)
majority_label = int(train_tickets["label"].mode().iloc[0])
majority_predictions = [majority_label] * len(true_labels)
comparison_rows = []
for name, predictions in [("Majority class", majority_predictions), ("Feature Engineering + MLP", count_predictions), ("BERT (DistilBERT)", predicted_labels), ("GPT (GPT-2)", gpt_predictions)]:
    comparison_rows.append({
        "model": name, "accuracy": accuracy_score(true_labels, predictions),
        "macro_f1": f1_score(true_labels, predictions, labels=list(id_to_label), average="macro", zero_division=0),
    })
print(pd.DataFrame(comparison_rows).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, name, predictions in [(axes[0], "Feature Engineering + MLP", count_predictions), (axes[1], "BERT (DistilBERT)", predicted_labels), (axes[2], "GPT (GPT-2)", gpt_predictions)]:
    print(name)
    print(classification_report(true_labels, predictions, labels=list(id_to_label), target_names=class_names, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(
        true_labels, predictions, labels=list(id_to_label), display_labels=class_names,
        xticks_rotation=30, cmap="Blues", colorbar=False, ax=ax,
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()
ticket_results = pd.DataFrame({
    "text": test_tickets["text"].tolist(),
    "actual": [id_to_label[int(label)] for label in true_labels],
    "word_counts_nn": [id_to_label[int(label)] for label in count_predictions],
    "distilbert": [id_to_label[int(label)] for label in predicted_labels],
    "gpt2": [id_to_label[int(label)] for label in gpt_predictions],
})
print("Tickets misclassified by any method:")
prediction_columns = ["word_counts_nn", "distilbert", "gpt2"]
errors = ticket_results[prediction_columns].ne(ticket_results["actual"], axis=0).any(axis=1)
print(ticket_results[errors].to_string(index=False))


## 6. Save, Reload and Route New Tickets

A softmax score is relative to the model's available classes. It is not a calibrated probability that the routing decision is correct. Out-of-scope tickets and ambiguous requests need a review process.

Save the count vocabulary together with the fitted scikit-learn MLP: changing the vocabulary order changes what each input column means. Save each Transformer tokenizer together with its model, including the GPT-2 padding configuration, so that inference uses the same input encoding.


In [ ]:
import os
import joblib
from transformers import pipeline

COUNT_SAVE_DIR = "transformer/saved-count-mlp"
os.makedirs(COUNT_SAVE_DIR, exist_ok=True)
# Keep the vocabulary, classifier and label names together for consistent inference.
joblib.dump({
    "vectorizer": count_vectorizer, "classifier": count_model, "class_names": class_names,
}, COUNT_SAVE_DIR + "/classifier.joblib")
saved_count = joblib.load(COUNT_SAVE_DIR + "/classifier.joblib")
reloaded_vectorizer = saved_count["vectorizer"]
reloaded_count_model = saved_count["classifier"]

TICKET_SAVE_DIR = "transformer/saved-ticket-classifier"
os.makedirs(TICKET_SAVE_DIR, exist_ok=True)
ticket_trainer.save_model(TICKET_SAVE_DIR)
ticket_tokenizer.save_pretrained(TICKET_SAVE_DIR)
ticket_classifier = pipeline("text-classification", model=TICKET_SAVE_DIR, tokenizer=TICKET_SAVE_DIR)
GPT_SAVE_DIR = "transformer/saved-gpt-classifier"
os.makedirs(GPT_SAVE_DIR, exist_ok=True)
gpt_trainer.save_model(GPT_SAVE_DIR)
gpt_tokenizer.save_pretrained(GPT_SAVE_DIR)
gpt_classifier = pipeline("text-classification", model=GPT_SAVE_DIR, tokenizer=GPT_SAVE_DIR)

new_tickets = [
    "My laptop cannot connect to campus Wi-Fi.",
    "I dropped my monitor and the display is broken.",
    "The program crashes, but only when I connect to Wi-Fi.",
]
new_counts = reloaded_vectorizer.transform(new_tickets)
count_classes = reloaded_count_model.predict(new_counts)
bert_results = ticket_classifier(new_tickets, truncation=True, max_length=MAX_TICKET_TOKENS)
gpt_results = gpt_classifier(new_tickets, truncation=True, max_length=MAX_TICKET_TOKENS)
for text, count_class, bert_result, gpt_result in zip(new_tickets, count_classes, bert_results, gpt_results):
    print(text)
    print("Feature Engineering + MLP:", saved_count["class_names"][count_class])
    print("BERT (DistilBERT):", bert_result, "| GPT (GPT-2):", gpt_result)
